## Kaggle prerequisites (do this first)

1. **Settings → Accelerator → GPU** (T4 / P100). CPU will **not** run Unsloth training.
2. **Settings → Internet → On** (required for `git clone`, pip from git, and Hugging Face model download).
3. **Secrets:** Add **`OPENAI_API_KEY`** in **Add-ons → Secrets** (attach to this notebook). The next cell loads it into the environment.
4. Optional: fork users can set a **Kaggle environment variable** `EPISTEMICOPS_BRANCH` to your branch name before the clone cell.

**After a long baseline run:** use **Session → Restart session** before the Unsloth model cell (same memory pattern as Colab).


In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
    _k = _sec.get_secret("OPENAI_API_KEY")
    if _k:
        os.environ["OPENAI_API_KEY"] = _k
except Exception as e:
    print("Secrets not available or key missing:", e)
print("OPENAI_API_KEY loaded:", bool(os.environ.get("OPENAI_API_KEY")))


# EpistemicOps: GRPO Training with Unsloth & TRL (Kaggle)

Same pipeline as the Colab notebook (kept as `training/colab_grpo_training.ipynb`):

1. Baseline episodes (mock primary, no LLM)
2. **Restart session** to free memory before loading the 8B model
3. Load Llama 3.1 8B (4-bit) via Unsloth
4. GRPO with `epistemicops_reward_function`
5. Episode-level proof (`eval/proof_of_learning.py`) and plots


In [ ]:
# Pre-pip: confirm NVIDIA driver visible (does not import torch)
!nvidia-smi -L || true


In [ ]:
# Installs: do not import torch before this cell on Kaggle (Unsloth/TRL pip order).
!pip install -q openai anthropic
!pip install -q "transformers>=4.51.3,<=5.5.0" trl datasets httpx pydantic pyyaml matplotlib numpy
!pip install -q "unsloth[kaggle] @ git+https://github.com/unslothai/unsloth.git"


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Enable GPU: Settings → Accelerator → GPU, then restart session."
    )
print("CUDA OK:", torch.cuda.get_device_name(0))


In [ ]:
import os, subprocess

REPO_DIR = "/kaggle/working/EpistemicOPS"
TARGET_BRANCH = os.getenv("EPISTEMICOPS_BRANCH", "master")

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/divyam-r25/EpistemicOPS.git", REPO_DIR], check=True)

subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "--prune"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", TARGET_BRANCH], check=True)
subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", TARGET_BRANCH], check=True)

os.chdir(REPO_DIR)
os.environ["EPISTEMICOPS_OFFLINE"] = "true"

os.environ.setdefault("JUDGE_PROVIDER", "openai")
os.environ.setdefault("JUDGE_MODEL", "gpt-4o-mini")
os.environ.setdefault("PRIMARY_AGENT_MODEL", "gpt-4o-mini")

commit = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"]).decode().strip()
print(f"Working dir: {os.getcwd()}")
print(f"Checked out branch: {TARGET_BRANCH}")
print(f"Commit: {commit}")
print("Model defaults: PRIMARY_AGENT_MODEL=gpt-4o-mini, JUDGE_PROVIDER=openai, JUDGE_MODEL=gpt-4o-mini")
print("Restart session after baseline if you want a clean kernel before Unsloth.")


---
## Step 1: Baseline evaluation (before training)

Mock primary agent (`primary_use_llm=False`). Saves `baseline_results.json` under the repo root (persists in `/kaggle/working` for the session).


In [ ]:
import asyncio, json, sys, random, inspect, os
sys.path.insert(0, ".")

from run_episode import run_full_episode

sig = inspect.signature(run_full_episode)
accepted_params = list(sig.parameters.keys())
print("run_full_episode params:", accepted_params)
required_params = {"primary_profile", "primary_use_llm"}
if not required_params.issubset(set(accepted_params)):
    raise RuntimeError(
        "Stale run_episode.py. Restart session, rerun clone cell, then this cell. "
        f"Missing: {sorted(required_params - set(accepted_params))}"
    )

EVAL_SCENARIOS = ["cascading_incident"]
EVAL_RUNS_PER_SCENARIO = 10
EVAL_ERAS_PER_RUN = 2
SEED_BASE = 7

baseline_rewards = []
baseline_records = []
for scenario_id in EVAL_SCENARIOS:
    for i in range(EVAL_RUNS_PER_SCENARIO):
        random.seed(SEED_BASE + i * 42)
        episode = await run_full_episode(
            scenario_id=scenario_id,
            num_eras=EVAL_ERAS_PER_RUN,
            primary_profile="baseline",
            primary_use_llm=False,
        )
        avg_r = episode["avg_normalized_reward"]
        baseline_rewards.append(avg_r)
        baseline_records.append({
            "scenario_id": scenario_id,
            "run_idx": i + 1,
            "avg_normalized_reward": avg_r,
        })
        print(f"[{scenario_id}] Episode {i+1}: R_normalized = {avg_r:.4f}")

baseline_mean = sum(baseline_rewards) / len(baseline_rewards)
print()
print(f"Baseline mean: {baseline_mean:.4f}")

with open("baseline_results.json", "w") as f:
    json.dump({
        "baseline_rewards": baseline_rewards,
        "baseline_records": baseline_records,
        "mean": baseline_mean,
        "eval_config": {
            "scenarios": EVAL_SCENARIOS,
            "runs_per_scenario": EVAL_RUNS_PER_SCENARIO,
            "eras_per_run": EVAL_ERAS_PER_RUN,
            "seed_base": SEED_BASE,
        },
    }, f, indent=2)
print("Saved to baseline_results.json")


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#16213e")
ax.bar(range(1, len(baseline_rewards)+1), baseline_rewards,
       color="#e94560", alpha=0.85, label="Baseline (mock agent)")
ax.axhline(y=np.mean(baseline_rewards), color="#FFD700",
           linestyle="--", linewidth=2, label=f"Mean: {np.mean(baseline_rewards):.3f}")
ax.set_xlabel("Episode", color="white", fontsize=12)
ax.set_ylabel("Normalized Reward", color="white", fontsize=12)
ax.set_title("Baseline performance (no training)", color="white", fontsize=14, fontweight="bold")
ax.set_ylim(0, 1.0)
ax.tick_params(colors="white")
ax.legend(facecolor="#1a1a2e", edgecolor="#444", labelcolor="white")
ax.grid(axis="y", alpha=0.15, color="white")
plt.tight_layout()
os.makedirs("plots", exist_ok=True)
plt.savefig("plots/kaggle_baseline.png", dpi=150, facecolor="#1a1a2e")
plt.show()
print("Baseline plot saved to plots/kaggle_baseline.png")


---
## Restart session (important)

Baseline pulls in environment + agent code. **Session → Restart session** to free RAM before loading the 8B model.

After restart: **skip** install/clone if outputs still exist, or re-run from the **clone** cell through **baseline** if `baseline_results.json` is missing.

Then continue with **Step 2** below.


---
## Step 2: Load model (after restart)

Run from `/kaggle/working/EpistemicOPS` with a clean GPU. Prefer running **install → clone → this section** without importing `training.train_primary` before `import unsloth` (avoids Unsloth import-order warnings).


In [ ]:
import os, sys, json

os.chdir("/kaggle/working/EpistemicOPS")
sys.path.insert(0, ".")
os.environ["EPISTEMICOPS_OFFLINE"] = "true"

with open("baseline_results.json") as f:
    saved = json.load(f)

baseline_rewards = saved["baseline_rewards"]
baseline_records = saved.get("baseline_records", [])
eval_config = saved.get("eval_config", {
    "scenarios": ["cascading_incident"],
    "runs_per_scenario": 10,
    "eras_per_run": 2,
    "seed_base": 7,
})

print(f"Loaded baseline: mean = {saved['mean']:.4f} ({len(baseline_rewards)} episodes)")
print("Evaluation config:", eval_config)


In [ ]:
# Unsloth: import package first (recommended), then FastLanguageModel
import unsloth  # noqa: F401
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=1024,
    load_in_4bit=True,
    fast_inference=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Model loaded. Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")


---
## Step 3: Prompt dataset and reward check


In [ ]:
from training.train_primary import build_prompt_dataset, epistemicops_reward_function

dataset = build_prompt_dataset(num_samples=200)
print(f"Dataset: {len(dataset)} prompts")
print("Sample (first 200 chars):")
print(dataset[0]["prompt"][:200] + "...")


In [ ]:
test_actions = [
    '{"action_type": "call_tool", "payload": {"tool": "get_incident_status", "args": {"incident_id": "INC-2041"}}}',
    '{"action_type": "declare_hypothesis", "payload": {"hypothesis": "API drift detected", "confidence": 0.7}}',
    '{"action_type": "write_legacy", "payload": {"content": "SECTION 1: state\nSECTION 2: trust\nSECTION 3: drift\nSECTION 4: decisions\nSECTION 5: issues\nSECTION 6: actions"}}',
    "invalid json garbage",
    '{"action_type": "hallucinated_tool", "payload": {}}',
]
rewards = epistemicops_reward_function(test_actions)
for action, r in zip(test_actions, rewards):
    print(f"  R={r:.2f}  {action[:70]}")
print()
print(f"Reward range: [{min(rewards):.2f}, {max(rewards):.2f}]")


---
## Step 4: GRPO training


In [ ]:
from trl import GRPOTrainer, GRPOConfig

# Recent TRL: GRPOConfig dropped the legacy prompt-length cap; prompts are bounded in build_prompt_dataset (legacy_doc[:800]) + tokenizer.
training_args = GRPOConfig(
    output_dir="./checkpoints/primary_agent",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    beta=0.1,
    temperature=0.8,
    num_generations=2,
    max_completion_length=256,
    logging_steps=1,
    save_steps=50,
    bf16=False,
    fp16=True,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=epistemicops_reward_function,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Starting GRPO training...")
train_result = trainer.train()
print(f"Training complete. Loss: {train_result.training_loss:.4f}")


In [ ]:
model.save_pretrained("./checkpoints/primary_agent_final")
tokenizer.save_pretrained("./checkpoints/primary_agent_final")
print("Model checkpoint saved to ./checkpoints/primary_agent_final")


---
## Step 5: Results and proof


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

logs = trainer.state.log_history
train_steps = [l["step"] for l in logs if "loss" in l]
train_losses = [l["loss"] for l in logs if "loss" in l]

fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#16213e")
ax.plot(train_steps, train_losses, color="#00d4ff", linewidth=2, marker="o", markersize=4)
ax.set_xlabel("Training Step", color="white", fontsize=12)
ax.set_ylabel("Loss", color="white", fontsize=12)
ax.set_title("GRPO training loss", color="white", fontsize=14, fontweight="bold")
ax.tick_params(colors="white")
ax.grid(alpha=0.15, color="white")
plt.tight_layout()
plt.savefig("plots/training_loss.png", dpi=150, facecolor="#1a1a2e")
plt.show()


In [ ]:
import json
import subprocess
from pathlib import Path

proof_cmd = [
    "python",
    "eval/proof_of_learning.py",
    "--scenarios", ",".join(eval_config["scenarios"]),
    "--runs-per-scenario", str(eval_config["runs_per_scenario"]),
    "--eras-per-run", str(eval_config["eras_per_run"]),
    "--baseline-profile", "baseline",
    "--trained-agent-source", "checkpoint",
    "--trained-checkpoint-path", "./checkpoints/primary_agent_final",
]

print("Running proof command:")
print(" ".join(proof_cmd))
subprocess.run(proof_cmd, check=True)

proof_path = Path("eval_results/proof_of_learning.json")
with proof_path.open() as f:
    proof = json.load(f)

summary = proof["summary"]
baseline_mean = summary["baseline"]["avg_reward"]
trained_mean = summary["trained"]["avg_reward"]
print()
print("Loaded proof summary:")
print(json.dumps(summary, indent=2))
print()
print(f"Baseline mean reward:  {baseline_mean:.4f}")
print(f"Trained mean reward:   {trained_mean:.4f}")
print(f"Improvement:           {trained_mean - baseline_mean:+.4f}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor("#1a1a2e")

ax1.set_facecolor("#16213e")
metric_keys = ["avg_reward", "avg_criteria_completion", "drift_detection_rate", "incident_resolution_rate", "legacy_doc_rate"]
metric_labels = ["Avg Reward", "Criteria", "Drift Detect", "Incident Resolve", "Legacy Doc"]
baseline_vals = [summary["baseline"][k] for k in metric_keys]
trained_vals = [summary["trained"][k] for k in metric_keys]

x = np.arange(len(metric_labels))
w = 0.35
ax1.bar(x - w/2, baseline_vals, w, label="Baseline", color="#e94560", alpha=0.85)
ax1.bar(x + w/2, trained_vals, w, label="Post-GRPO", color="#00d4ff", alpha=0.9)
ax1.set_xticks(x)
ax1.set_xticklabels(metric_labels, rotation=15)
ax1.set_ylim(0, 1.05)
ax1.set_title("Episode-level metric comparison", color="white", fontsize=13, fontweight="bold")
ax1.tick_params(colors="white")
ax1.legend(facecolor="#1a1a2e", edgecolor="#444", labelcolor="white")
ax1.grid(axis="y", alpha=0.15, color="white")

ax2.set_facecolor("#16213e")
means = [baseline_mean, trained_mean]
bars = ax2.bar(["Baseline", "Post-GRPO"], means, color=["#e94560", "#00d4ff"], alpha=0.9)
for bar, m in zip(bars, means):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{m:.3f}", ha="center", color="white", fontsize=14, fontweight="bold")
ax2.set_ylabel("Avg normalized reward", color="white", fontsize=12)
ax2.set_title("Mean reward (same env)", color="white", fontsize=13, fontweight="bold")
ax2.set_ylim(0, 1.0)
ax2.tick_params(colors="white")
ax2.grid(axis="y", alpha=0.15, color="white")

fig.suptitle("EpistemicOps: baseline vs GRPO checkpoint (episode-level)",
             color="white", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/before_after_comparison.png", dpi=150, facecolor="#1a1a2e")
plt.show()
print("Comparison plot saved.")


In [ ]:
import json
import os
import platform
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
import numpy
import pydantic
import yaml

metadata = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "notebook": "kaggle_grpo_training",
    "offline_mode": os.getenv("EPISTEMICOPS_OFFLINE", ""),
    "eval_config": eval_config,
    "proof_config": proof.get("config", {}),
    "checkpoint_path": "./checkpoints/primary_agent_final",
    "summary": summary,
    "deltas": proof.get("deltas", {}),
    "package_versions": {
        "matplotlib": matplotlib.__version__,
        "numpy": numpy.__version__,
        "pydantic": pydantic.__version__,
        "pyyaml": yaml.__version__,
    },
}

for pkg_name in ["trl", "transformers", "datasets", "torch"]:
    try:
        module = __import__(pkg_name)
        metadata["package_versions"][pkg_name] = getattr(module, "__version__", "unknown")
    except Exception:
        metadata["package_versions"][pkg_name] = "not_installed"

try:
    metadata["git_commit"] = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
except Exception:
    metadata["git_commit"] = "unknown"

try:
    metadata["gpu_name"] = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]
    ).decode().strip()
except Exception:
    metadata["gpu_name"] = "unknown"

meta_path = Path("eval_results/proof_run_metadata.json")
meta_path.parent.mkdir(parents=True, exist_ok=True)
meta_path.write_text(json.dumps(metadata, indent=2))
print(f"Saved run metadata to {meta_path}")

behavior_path = Path("eval_results/proof_behavior_examples.md")
if behavior_path.exists():
    behavior_text = behavior_path.read_text()
    print()
    print("=== Behavior evidence (compact) ===")
    for line in behavior_text.splitlines()[:26]:
        print(line)
else:
    print("Behavior evidence file missing:", behavior_path)


---
## Summary (judge-facing)

Episode-level baseline vs trained comparison:

`python eval/proof_of_learning.py --trained-agent-source checkpoint --trained-checkpoint-path ./checkpoints/primary_agent_final`

Artifacts: `eval_results/proof_of_learning.json`, `proof_run_metadata.json`, `proof_behavior_examples.md`, `plots/proof_*.png`, `plots/before_after_comparison.png`.

**Save outputs:** Kaggle → **Save Version** (Save & Run All) or download files from `/kaggle/working/EpistemicOPS/`.

Colab backup: [`training/colab_grpo_training.ipynb`](https://github.com/divyam-r25/EpistemicOPS/blob/master/training/colab_grpo_training.ipynb) | [HF Space](https://huggingface.co/spaces/Divyam-r25/EpistemicOps) | [GitHub](https://github.com/divyam-r25/EpistemicOPS)


### Optional: checkpoint-backed episodes

```bash
python training/record_checkpoint_episodes.py --checkpoint ./checkpoints/primary_agent_final --runs 3 --eras 3
```
